# TorchRL Q-values

In this tutorial, we will start to allow our agents to learn!

This notebook was modified from the official tutorials provided by torchRL: 

**Get started with TorchRL's modules**
Author: `Vincent Moens <https://github.com/vmoens>`
url: https://docs.pytorch.org/rl/stable/tutorials/getting-started-1.html

### Install TorchRL

First let's install torchrl and tensordict.


In [ ]:
#!pip install tensordict
#!pip install torchrl

In [86]:
from tensordict.nn import TensorDictSequential
from torchrl.modules import EGreedyModule

import torch
from tensordict.nn import TensorDictModule
from tensordict import TensorDict
from torchrl.envs import GymEnv
from torch import nn

# Q-values

In [4]:
env = GymEnv("CartPole-v1")

#Actions
env_actions = env.action_spec.shape[-1]
print(f"Action space: {env_actions}")

#Actions
env_obs = env.observation_spec["observation"].shape[-1]
print(f"Observation space: {env_obs}")

Action space: 2
Observation space: 4


In [ ]:
from torchrl.modules import ProbabilisticActor
from torchrl.modules.distributions import OneHotCategorical

#create a model: takes observations as inputs, and outputs categorical actions
model = nn.Sequential(
    nn.Linear(env_obs,64),
    nn.ReLU(),
    nn.Linear(64,64),
    nn.ReLU(),
    nn.Linear(64, env_actions),
)

#go from observations to logits using the model
value_net = TensorDictModule(
    model,
    in_keys=["observation"],
    out_keys=["action_value"],          
)


In [6]:
from torchrl.modules import QValueModule

policy = TensorDictSequential(
    value_net,  
    QValueModule(spec=env.action_spec),  #expects the action_value output from value_net
)

In [7]:
rollout = env.rollout(max_steps=3, policy=policy)
print(rollout)

TensorDict(
    fields={
        action: Tensor(shape=torch.Size([3, 2]), device=cpu, dtype=torch.int64, is_shared=False),
        action_value: Tensor(shape=torch.Size([3, 2]), device=cpu, dtype=torch.float32, is_shared=False),
        chosen_action_value: Tensor(shape=torch.Size([3, 1]), device=cpu, dtype=torch.float32, is_shared=False),
        done: Tensor(shape=torch.Size([3, 1]), device=cpu, dtype=torch.bool, is_shared=False),
        next: TensorDict(
            fields={
                done: Tensor(shape=torch.Size([3, 1]), device=cpu, dtype=torch.bool, is_shared=False),
                observation: Tensor(shape=torch.Size([3, 4]), device=cpu, dtype=torch.float32, is_shared=False),
                reward: Tensor(shape=torch.Size([3, 1]), device=cpu, dtype=torch.float32, is_shared=False),
                terminated: Tensor(shape=torch.Size([3, 1]), device=cpu, dtype=torch.bool, is_shared=False),
                truncated: Tensor(shape=torch.Size([3, 1]), device=cpu, dtype=torch

We can see that the QValueModule is adding some new data to our TED. It now has:

* action_value: the estimated value of an action

In [8]:
rollout["action_value"]

tensor([[0.1705, 0.1609],
        [0.1614, 0.1474],
        [0.1557, 0.1355]], grad_fn=<StackBackward0>)

In [9]:
rollout["chosen_action_value"]

tensor([[0.1705],
        [0.1614],
        [0.1557]], grad_fn=<StackBackward0>)

It's choosing the action with the estimated best value. 

But let's let it explore too

In [10]:
from torchrl.envs.utils import ExplorationType, set_exploration_type

exploration_module = EGreedyModule(
    spec=env.action_spec, 
    annealing_num_steps=1000, 
    eps_init=0.95
)

policy_explore = TensorDictSequential(policy, exploration_module)

with set_exploration_type(ExplorationType.RANDOM):
    rollout_explore = env.rollout(max_steps=10, policy=policy_explore)

rollout_explore

TensorDict(
    fields={
        action: Tensor(shape=torch.Size([10, 2]), device=cpu, dtype=torch.int64, is_shared=False),
        action_value: Tensor(shape=torch.Size([10, 2]), device=cpu, dtype=torch.float32, is_shared=False),
        chosen_action_value: Tensor(shape=torch.Size([10, 1]), device=cpu, dtype=torch.float32, is_shared=False),
        done: Tensor(shape=torch.Size([10, 1]), device=cpu, dtype=torch.bool, is_shared=False),
        next: TensorDict(
            fields={
                done: Tensor(shape=torch.Size([10, 1]), device=cpu, dtype=torch.bool, is_shared=False),
                observation: Tensor(shape=torch.Size([10, 4]), device=cpu, dtype=torch.float32, is_shared=False),
                reward: Tensor(shape=torch.Size([10, 1]), device=cpu, dtype=torch.float32, is_shared=False),
                terminated: Tensor(shape=torch.Size([10, 1]), device=cpu, dtype=torch.bool, is_shared=False),
                truncated: Tensor(shape=torch.Size([10, 1]), device=cpu, dt

In [11]:
rollout_explore["action_value"]

tensor([[0.1620, 0.1615],
        [0.1625, 0.1507],
        [0.1622, 0.1616],
        [0.1530, 0.1547],
        [0.1436, 0.1376],
        [0.1434, 0.1372],
        [0.1451, 0.1309],
        [0.1500, 0.1214],
        [0.1596, 0.1030],
        [0.1732, 0.0874]], grad_fn=<StackBackward0>)

In [12]:
rollout_explore["action"]

tensor([[0, 1],
        [1, 0],
        [1, 0],
        [1, 0],
        [1, 0],
        [1, 0],
        [1, 0],
        [1, 0],
        [1, 0],
        [1, 0]])

You should now see that the agent is exploring actions, and not just chosing the highest value action.

### Actor Modules

We can also do the same as above but take advantage of actor modules. These modules handle common action selection patterns. Let's take a look at doing the same thing as above but just with the QvalueActor module.



In [29]:
from torchrl.modules import QValueActor

#create a model: takes observations as inputs, and outputs categorical actions
model = nn.Sequential(
    nn.Linear(env_obs,64),
    nn.ReLU(),
    nn.Linear(64,64),
    nn.ReLU(),
    nn.Linear(64, env_actions),
)

#go from observations to logits using the model
value_net = TensorDictModule(
    model,
    in_keys=["observation"],
    out_keys=["action_value"],          
)

qvalue_actor = QValueActor(module=value_net, spec=env.action_spec)

Now we can use this actor with the environment

In [ ]:
with set_exploration_type(ExplorationType.RANDOM):
    rollout_qvalue_actor = env.rollout(max_steps=10, policy=qvalue_actor)


print(rollout_qvalue_actor)

TensorDict(
    fields={
        action: Tensor(shape=torch.Size([10, 2]), device=cpu, dtype=torch.int64, is_shared=False),
        action_value: Tensor(shape=torch.Size([10, 2]), device=cpu, dtype=torch.float32, is_shared=False),
        chosen_action_value: Tensor(shape=torch.Size([10, 1]), device=cpu, dtype=torch.float32, is_shared=False),
        done: Tensor(shape=torch.Size([10, 1]), device=cpu, dtype=torch.bool, is_shared=False),
        next: TensorDict(
            fields={
                done: Tensor(shape=torch.Size([10, 1]), device=cpu, dtype=torch.bool, is_shared=False),
                observation: Tensor(shape=torch.Size([10, 4]), device=cpu, dtype=torch.float32, is_shared=False),
                reward: Tensor(shape=torch.Size([10, 1]), device=cpu, dtype=torch.float32, is_shared=False),
                terminated: Tensor(shape=torch.Size([10, 1]), device=cpu, dtype=torch.bool, is_shared=False),
                truncated: Tensor(shape=torch.Size([10, 1]), device=cpu, dt

In [ ]:
rollout_qvalue_actor["action"]

tensor([[1, 0],
        [1, 0],
        [0, 1],
        [1, 0],
        [0, 1],
        [1, 0],
        [0, 1],
        [1, 0],
        [0, 1],
        [1, 0]])

In [ ]:
rollout_qvalue_actor["action_value"]

tensor([[0.0947, 0.0557],
        [0.0818, 0.0660],
        [0.0735, 0.0890],
        [0.0809, 0.0664],
        [0.0729, 0.0906],
        [0.0798, 0.0675],
        [0.0723, 0.0935],
        [0.0785, 0.0700],
        [0.0723, 0.0983],
        [0.0770, 0.0736]], grad_fn=<StackBackward0>)

Let's make it so that the agent can explore actions as well

In [38]:
exploration_module = EGreedyModule(
    spec=env.action_spec, 
    annealing_num_steps=1000, 
    eps_init=0.95
)

qvalue_actor_explore = TensorDictSequential(qvalue_actor, exploration_module)

In [46]:
with set_exploration_type(ExplorationType.RANDOM):
    rollout_qvalue_actor_explore = env.rollout(max_steps=10, policy=qvalue_actor_explore)

print(rollout_qvalue_actor_explore["action"])
print(rollout_qvalue_actor_explore["action_value"])

tensor([[0, 1],
        [0, 1],
        [0, 1],
        [0, 1],
        [0, 1],
        [1, 0],
        [1, 0],
        [1, 0],
        [1, 0],
        [1, 0]])
tensor([[0.0939, 0.0565],
        [0.1081, 0.0553],
        [0.1308, 0.0682],
        [0.1483, 0.0659],
        [0.1521, 0.0588],
        [0.1581, 0.0526],
        [0.1581, 0.0585],
        [0.1579, 0.0654],
        [0.1419, 0.0646],
        [0.1192, 0.0567]], grad_fn=<StackBackward0>)


You shoul now see the agent exploring actions as well as exploiting highest value actions.

### Deep q-learning (DQN)

Now that we have this working let's dig into how to use these action values to allow our agent to learn.

In TorchRL, we try to treat optimization as it is custom to do in PyTorch, using dedicated loss modules which are designed with the sole purpose of optimizing the model. This approach efficiently decouples the execution of the policy from its training and allows us to design training loops that are similar to what can be found in traditional supervised learning examples.

The typical training loop therefore looks like this:

```
for i in range(n_data_collections):
        data = get_next_data_batch(env, policy)
        for j in range(n_optim):
            loss = loss_fn(data)
            loss.backward()
            optim.step()
```

In RL, innovation typically involves the exploration of novel methods for optimizing a policy (i.e., new algorithms), rather than focusing on new architectures, as seen in other domains. Within TorchRL, these algorithms are encapsulated within loss modules. A loss module orchestrates the various components of your algorithm and yields a set of loss values that can be backpropagated through to train the corresponding components.

In this tutorial, we will take a popular off-policy algorithm as an example, DQN.



To build a loss module, the only thing one needs is a set of networks defined as TensorDictModule. Most of the time, one of these modules will be the policy. 

Let’s see what this looks like in practice: DQN requires a deterministic map from the observation space to the action space as well as a value network that predicts the value of a state-action pair. The DQN loss will attempt to find the policy parameters that output actions that maximize the value for a given state.

To build the loss, we need both the actor and value networks. If they are built according to DQN’s expectations, it is all we need to get a trainable loss module:

In [ ]:

env = GymEnv("CartPole-v1")

from torchrl.modules import Actor, MLP, ValueOperator
from torchrl.objectives import DQNLoss

n_obs = env.observation_spec["observation"].shape[-1]
n_act = env.action_spec.shape[-1]


#create a model: takes observations as inputs, and outputs categorical actions
model = nn.Sequential(
    nn.Linear(env_obs,64),
    nn.ReLU(),
    nn.Linear(64,64),
    nn.ReLU(),
    nn.Linear(64, env_actions),
)

#go from observations to logits using the model
value_net = TensorDictModule(
    model,
    in_keys=["observation"],
    out_keys=["action_value"],          
)

qvalue_actor = QValueActor(module=value_net, spec=env.action_spec)

DQN_loss = DQNLoss(qvalue_actor) #when updating use greedy (always choose best action)

In [ ]:
#when collecting data sometimes use random (e-greedy)
exploration_module = EGreedyModule(
    spec=env.action_spec, 
    annealing_num_steps=1000, 
    eps_init=0.95
)
qvalue_actor_explore = TensorDictSequential(qvalue_actor, exploration_module)

rollout = env.rollout(max_steps=100, policy=qvalue_actor_explore)




TensorDict(
    fields={
        loss: Tensor(shape=torch.Size([]), device=cpu, dtype=torch.float32, is_shared=False)},
    batch_size=torch.Size([]),
    device=None,
    is_shared=False)


/home/titan2/data/conda-envs/torchrl_latest/lib/python3.12/site-packages/torchrl/objectives/common.py:40: UserWarning: No target network updater has been associated with this loss module, but target parameters have been found. While this is supported, it is expected that the target network updates will be manually performed. You can deactivate this warning by turning the RL_WARNINGS env variable to False.
  warnings.warn(
/home/titan2/data/conda-envs/torchrl_latest/lib/python3.12/site-packages/torchrl/objectives/common.py:457: UserWarning: No target network updater has been associated with this loss module, but target parameters have been found. While this is supported, it is expected that the target network updates will be manually performed. You can deactivate this warning by turning the RL_WARNINGS env variable to False.
  warnings.warn(


In [75]:
rollout["action"]

tensor([[1, 0],
        [1, 0],
        [1, 0],
        [1, 0],
        [0, 1],
        [1, 0],
        [0, 1],
        [1, 0],
        [0, 1],
        [1, 0],
        [0, 1],
        [1, 0]])

Let's calculate the loss from some of the observed data

In [77]:
loss_vals = DQN_loss(rollout)
print(loss_vals)

TensorDict(
    fields={
        loss: Tensor(shape=torch.Size([]), device=cpu, dtype=torch.float32, is_shared=False)},
    batch_size=torch.Size([]),
    device=None,
    is_shared=False)


As you can see, the value we received from the loss isn’t a single scalar but a dictionary. This can be useful especially when the loss outputs multiple losses.



In [78]:
obs_loss = loss_vals["loss"]
print(obs_loss)


tensor(1.0266, grad_fn=<MeanBackward0>)


This is how far off our model is from predicting the observed value of actions. 

### Training a LossModule

Given all this, training the modules is not so different from what would be done in any other training loop. Because it wraps the modules, the easiest way to get the list of trainable parameters is to query the parameters() method.

We’ll need an optimizer (or one optimizer per module if that is your choice).

In [58]:
from torch.optim import Adam

optim = Adam(DQN_loss.parameters())
obs_loss.backward()

Once we run the obs_loss.backwards the parameters in the *model* will be updated to produce action values that match with what the agent observed. Remember that the agent after each step sees the next environmenal state and recives a reward after performing an action in their current state. The agent can then see if the predicted value of the current action matched with the recived rewards. If it did not, this backwards propogation of the loss should nudge the model to make a better prediction. 

Over many agent-environment interactions the aim is that the agent will be better at predicting which actions in which states lead to better outcomes.

### Manually calculate loss

Let's dig a little deeper to see if we can't better understand what the DQNLoss function is doing, and why it's doing it.

Let's see if we can calculate the loss ourselves. To do so let's:

* Take an action in the environment to collect some data
* Use outcome of the data to update our model to better predict the value of actions


In [100]:
rollout

TensorDict(
    fields={
        action: Tensor(shape=torch.Size([10, 2]), device=cpu, dtype=torch.int64, is_shared=False),
        action_value: Tensor(shape=torch.Size([10, 2]), device=cpu, dtype=torch.float32, is_shared=False),
        chosen_action_value: Tensor(shape=torch.Size([10, 1]), device=cpu, dtype=torch.float32, is_shared=False),
        done: Tensor(shape=torch.Size([10, 1]), device=cpu, dtype=torch.bool, is_shared=False),
        next: TensorDict(
            fields={
                done: Tensor(shape=torch.Size([10, 1]), device=cpu, dtype=torch.bool, is_shared=False),
                observation: Tensor(shape=torch.Size([10, 4]), device=cpu, dtype=torch.float32, is_shared=False),
                reward: Tensor(shape=torch.Size([10, 1]), device=cpu, dtype=torch.float32, is_shared=False),
                terminated: Tensor(shape=torch.Size([10, 1]), device=cpu, dtype=torch.bool, is_shared=False),
                truncated: Tensor(shape=torch.Size([10, 1]), device=cpu, dt

In [116]:
#agent-environment interactions!
rollout = env.rollout(max_steps=10, policy=qvalue_actor_explore)

#collect the data
obs    = rollout["observation"]
act    = rollout["action"].argmax(dim=-1) 
next_obs = rollout["next", "observation"]
reward = rollout["next", "reward"].squeeze(-1)
done   = rollout["next", "done"].squeeze(-1).float()

Feel free to take a look at what each bit of data looks like. E.g., we can see that the rewards are just 1s. 

In [117]:
rollout["action"]

tensor([[0, 1],
        [0, 1],
        [0, 1],
        [1, 0],
        [0, 1],
        [1, 0],
        [0, 1],
        [1, 0],
        [0, 1],
        [0, 1]])

Next let's calculate the value of current states and actions seen. Remember that this is not just the reward seen but the reward seen + the discounted value of the next state.

In [118]:
#place 
currrent_observations = TensorDict({"observation": obs}, batch_size=[obs.shape[0]])
currrent_observations

TensorDict(
    fields={
        observation: Tensor(shape=torch.Size([10, 4]), device=cpu, dtype=torch.float32, is_shared=False)},
    batch_size=torch.Size([10]),
    device=None,
    is_shared=False)

In [119]:
current_value = value_net(currrent_observations) #calculate value of current state, action
print(current_value["action_value"])

tensor([[ 0.0291, -0.0401],
        [ 0.0294, -0.0213],
        [ 0.0405, -0.0197],
        [ 0.0598, -0.0365],
        [ 0.0420, -0.0191],
        [ 0.0632, -0.0383],
        [ 0.0438, -0.0197],
        [ 0.0669, -0.0419],
        [ 0.0465, -0.0209],
        [ 0.0712, -0.0471]], grad_fn=<AddmmBackward0>)


Next let's get the value of the action that was taken

In [120]:
#q value of current action
q_sa = current_value["action_value"].gather(1, act.unsqueeze(-1)).squeeze(-1)

print(q_sa)


tensor([-0.0401, -0.0213, -0.0197,  0.0598, -0.0191,  0.0632, -0.0197,  0.0669,
        -0.0209, -0.0471], grad_fn=<SqueezeBackward1>)


Let's now let's look at what happened next! We'll use the difference between what was predicted and what actually happend to update our agent.



In [125]:
next_observations = TensorDict({"observation": next_obs}, batch_size=[next_obs.shape[0]])

next_value = value_net(next_obs)

print(next_value)

tensor([[ 0.0294, -0.0213],
        [ 0.0405, -0.0197],
        [ 0.0598, -0.0365],
        [ 0.0420, -0.0191],
        [ 0.0632, -0.0383],
        [ 0.0438, -0.0197],
        [ 0.0669, -0.0419],
        [ 0.0465, -0.0209],
        [ 0.0712, -0.0471],
        [ 0.0894, -0.0754]], grad_fn=<AddmmBackward0>)


Let's assume that in the next state the agent will choose the best possible action (with e-greedy it might not, but we can be a little optimistic here...)

In [126]:
next_q = next_value.max(dim=-1).values #DNQ

print(next_q)

tensor([0.0294, 0.0405, 0.0598, 0.0420, 0.0632, 0.0438, 0.0669, 0.0465, 0.0712,
        0.0894], grad_fn=<MaxBackward0>)


Now we can calculate the TD target. 

In [128]:
gamma = 0.99

target = reward + gamma * (1.0 - done) * next_q

Then we can see how well our model did at predicting the TD target.

In [129]:
#loss is the MSE between the current q_sa and the target (based on the reward observed and the discounted future q_sa)
import torch.nn.functional as F

loss = F.mse_loss(q_sa, target)

print(loss)

tensor(1.1147, grad_fn=<MseLossBackward0>)


Then we can backpropogate the loss. Though before we do that let's look at this all together and better handle what we track for the backpropogation step, i.e., when finding the TD-error we don't need to have all the steps included in the backpropogation step. 

In [ ]:

#agent-environment interactions!
rollout = env.rollout(max_steps=10, policy=qvalue_actor_explore)

#collect the data
obs    = rollout["observation"]
act    = rollout["action"].argmax(dim=-1) 
next_obs = rollout["next", "observation"]
reward = rollout["next", "reward"].squeeze(-1)
done   = rollout["next", "done"].squeeze(-1).float()


#get current estimated value of state-action
currrent_observations = TensorDict({"observation": obs}, batch_size=[obs.shape[0]])
current_value = value_net(currrent_observations) 
q_sa = current_value["action_value"].gather(1, act.unsqueeze(-1)).squeeze(-1)

#calculate the td-target
with torch.no_grad():

    #get current estimated value of state-action in the next step
    next_observations = TensorDict({"observation": next_obs}, batch_size=[next_obs.shape[0]])
    next_value = value_net(next_obs)
    next_q = next_value.max(dim=-1).values #DNQ

    #then get the target (observed reward + discounted value of the next state)
    target = reward + gamma * (1.0 - done) * next_q

#calculate the difference between the estimated value of the current state and the td-target
loss = F.mse_loss(q_sa, target)

#let's now optimize the model used to estimate q-values
optim.zero_grad(set_to_none=True)
loss.backward() #backprop!
optim.step()


    

Now that we know how to setup DQN let's put it into practice in the next tutorial!

### End

**Things to try**

* Check out more loss [modules](https://docs.pytorch.org/rl/stable/reference/objectives.html#ref-objectives) 
* Look at it being implemented in another algorithm ([DDPG](https://docs.pytorch.org/rl/stable/tutorials/coding_ddpg.html#coding-ddpg))